***

Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
# pd.options.display.float_format = '{:.0f}'.format

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths
if user == 'jfontes':
    # Git
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

    # SharePoint
    path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
    path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'TIMS Data')
    path_main = os.path.join(path_sp, 'Data')
    path_out  = os.path.join(path_main, 'Safe Equitable Resilient Infrastructure', 'Safety')
    path_tims = os.path.join(path_out, 'TIMS')

    
path_code    = os.path.join(path_git, 'Data', 'TIMS')
path_config0 = os.path.join(path_git , 'config')
path_config  = os.path.join(path_code, 'config')


In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

***

Importing

***

TIMS does not have an API available.  Data were manually downloaded from here https://tims.berkeley.edu/ and stored on SharePoint

In [ ]:
year_start = 2012
year_end = 2021
years_to_import = range(year_start, year_end+1)

In [ ]:
categories = [cat for cat in os.listdir(path_tims) if '.xlsx' not in cat]
categories

Jurisdictions

In [ ]:
# Import data at the jurisdictions level
exec(open(os.path.join(path_code, 'Supplemental Scripts', 'Jurisdictions.py')).read())
df_tims1.head()

In [ ]:
# Subset all TIMS data into different categories
list_id = ['County', 'Jurisdiction', 'Year']

df_tims1_fat = df_tims1[list_id + [              'Fatalities',               'Fatalities_5 Year Rolling Average']]
df_tims1_ser = df_tims1[list_id + [        'Serious injuries',         'Serious injuries_5 Year Rolling Average']]
df_tims1_non = df_tims1[list_id + ['Non motorized fatalities', 'Non motorized fatalities_5 Year Rolling Average'
                                 , 'Non motorized serious in', 'Non motorized serious in_5 Year Rolling Average']]

df_tims1_fat100 = df_tims1[list_id + [      'Fatalities 100 mvmt',      'Fatalities 100 mvmt_5 Year Rolling Average']]
df_tims1_ser100 = df_tims1[list_id + ['Serious injuries 100 mvm' , 'Serious injuries 100 mvm_5 Year Rolling Average']]

# Set columns of jurisdictions by counties
df_tims1_cols = df_tims1_fat.pivot_table(index = 'Year'
                                        , columns = ['County', 'Jurisdiction']
                                        , values = ['Fatalities', 'Fatalities_5 Year Rolling Average']).reset_index()
cols = [col[2] for col in df_tims1_cols.columns][1:]
cols = ['Year'] + cols

# Fatalities and Fatalities 100/MVMT
df_tims1_fat_2      = df_tims1_fat   .pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Fatalities'                                ).reset_index()
df_tims1_fat_2_5    = df_tims1_fat   .pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Fatalities_5 Year Rolling Average'         ).reset_index()
df_tims1_fat100_2   = df_tims1_fat100.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Fatalities 100 mvmt'                       ).reset_index()
df_tims1_fat100_2_5 = df_tims1_fat100.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Fatalities 100 mvmt_5 Year Rolling Average').reset_index()

# Serious Injuries and Serious Injuries 100/MVMT
df_tims1_ser_2      = df_tims1_ser   .pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Serious injuries'                               ).reset_index()
df_tims1_ser_2_5    = df_tims1_ser   .pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Serious injuries_5 Year Rolling Average'        ).reset_index()
df_tims1_ser100_2   = df_tims1_ser100.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Serious injuries 100 mvm'                       ).reset_index()
df_tims1_ser100_2_5 = df_tims1_ser100.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Serious injuries 100 mvm_5 Year Rolling Average').reset_index()

# Non-Motorized Fatalities and Serious Injuries
df_tims1_non_fat_2   = df_tims1_non.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Non motorized fatalities'                       ).reset_index()
df_tims1_non_fat_2_5 = df_tims1_non.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Non motorized fatalities_5 Year Rolling Average').reset_index()
df_tims1_non_si_2    = df_tims1_non.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Non motorized serious in'                       ).reset_index()
df_tims1_non_si_2_5  = df_tims1_non.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Non motorized serious in_5 Year Rolling Average').reset_index()

# Reorganize columns
df_tims1_fat_2       = df_tims1_fat_2      [cols]
df_tims1_fat_2_5     = df_tims1_fat_2_5    [cols]
df_tims1_fat100_2    = df_tims1_fat100_2   [cols]
df_tims1_fat100_2_5  = df_tims1_fat100_2_5 [cols]
df_tims1_ser_2       = df_tims1_ser_2      [cols]
df_tims1_ser_2_5     = df_tims1_ser_2_5    [cols]
df_tims1_ser100_2    = df_tims1_ser100_2   [cols]
df_tims1_ser100_2_5  = df_tims1_ser100_2_5 [cols]
df_tims1_non_fat_2   = df_tims1_non_fat_2  [cols]
df_tims1_non_fat_2_5 = df_tims1_non_fat_2_5[cols]
df_tims1_non_si_2    = df_tims1_non_si_2   [cols]
df_tims1_non_si_2_5  = df_tims1_non_si_2_5 [cols]

# View
display(df_tims1.head())

In [ ]:

# # Jurisdictions
# with pd.ExcelWriter(os.path.join(path_tims, 'TIMS SWITRS Data by Jurisdiction.xlsx'), engine='xlsxwriter') as writer:
#     df_tims1           .to_excel(writer, index = False, sheet_name = 'All'                            )
#     df_tims_fat_2      .to_excel(writer, index = False, sheet_name = 'Fatalities'                     )
#     df_tims_fat_2_5    .to_excel(writer, index = False, sheet_name = 'Fatalities 5 year'              )
#     df_tims_fat100_2   .to_excel(writer, index = False, sheet_name = 'Fatalities rate'                )
#     df_tims_fat100_2_5 .to_excel(writer, index = False, sheet_name = 'Fatalities rate 5 year'         )
#     df_tims_ser_2      .to_excel(writer, index = False, sheet_name = 'Serious injuries'               )
#     df_tims_ser_2_5    .to_excel(writer, index = False, sheet_name = 'Serious injuries 5 year'        )
#     df_tims_ser100_2   .to_excel(writer, index = False, sheet_name = 'Serious injuries rate'          )
#     df_tims_ser100_2_5 .to_excel(writer, index = False, sheet_name = 'Serious injuries rate 5 year'   )
#     df_tims_non_fat_2  .to_excel(writer, index = False, sheet_name = 'Non motorized fatalities'       )
#     df_tims_non_fat_2_5.to_excel(writer, index = False, sheet_name = 'Non motorized fatalities 5 year')
#     df_tims_non_si_2   .to_excel(writer, index = False, sheet_name = 'Non motorized serious in'       )
#     df_tims_non_si_2_5 .to_excel(writer, index = False, sheet_name = 'Non motorized serious in 5 year')


Counties

In [ ]:
# Import data at the counties level
exec(open(os.path.join(path_code, 'Supplemental Scripts', 'Counties.py')).read())
df_tims2.head()

In [ ]:

# Subset all TIMS data into different categories
list_id = ['County', 'Year']

df_tims2_fat = df_tims2[list_id + [              'Fatalities',               'Fatalities_5 Year Rolling Average']]
df_tims2_ser = df_tims2[list_id + [        'Serious injuries',         'Serious injuries_5 Year Rolling Average']]
df_tims2_non = df_tims2[list_id + ['Non motorized fatalities', 'Non motorized fatalities_5 Year Rolling Average'
                                 , 'Non motorized serious in', 'Non motorized serious in_5 Year Rolling Average']]

df_tims2_fat100 = df_tims2[list_id + [      'Fatalities 100 mvmt',      'Fatalities 100 mvmt_5 Year Rolling Average']]
df_tims2_ser100 = df_tims2[list_id + ['Serious injuries 100 mvm' , 'Serious injuries 100 mvm_5 Year Rolling Average']]

# Set columns of jurisdictions by counties
df_tims2_cols = df_tims2_fat.pivot_table(index = 'Year'
                                        , columns = 'County'
                                        , values = ['Fatalities', 'Fatalities_5 Year Rolling Average']).reset_index()
cols = [col[1] for col in df_tims2_cols.columns][1:]
cols = ['Year'] + cols

# Fatalities and Fatalities 100/MVMT
df_tims2_fat_2      = df_tims2_fat   .pivot_table(index = 'Year', columns = 'County', values = 'Fatalities'                                ).reset_index()
df_tims2_fat_2_5    = df_tims2_fat   .pivot_table(index = 'Year', columns = 'County', values = 'Fatalities_5 Year Rolling Average'         ).reset_index()
df_tims2_fat100_2   = df_tims2_fat100.pivot_table(index = 'Year', columns = 'County', values = 'Fatalities 100 mvmt'                       ).reset_index()
df_tims2_fat100_2_5 = df_tims2_fat100.pivot_table(index = 'Year', columns = 'County', values = 'Fatalities 100 mvmt_5 Year Rolling Average').reset_index()

# Serious Injuries and Serious Injuries 100/MVMT
df_tims2_ser_2      = df_tims2_ser   .pivot_table(index = 'Year', columns = 'County', values = 'Serious injuries'                               ).reset_index()
df_tims2_ser_2_5    = df_tims2_ser   .pivot_table(index = 'Year', columns = 'County', values = 'Serious injuries_5 Year Rolling Average'        ).reset_index()
df_tims2_ser100_2   = df_tims2_ser100.pivot_table(index = 'Year', columns = 'County', values = 'Serious injuries 100 mvm'                       ).reset_index()
df_tims2_ser100_2_5 = df_tims2_ser100.pivot_table(index = 'Year', columns = 'County', values = 'Serious injuries 100 mvm_5 Year Rolling Average').reset_index()

# Non-Motorized Fatalities and Serious Injuries
df_tims2_non_fat_2   = df_tims2_non.pivot_table(index = 'Year', columns = 'County', values = 'Non motorized fatalities'                       ).reset_index()
df_tims2_non_fat_2_5 = df_tims2_non.pivot_table(index = 'Year', columns = 'County', values = 'Non motorized fatalities_5 Year Rolling Average').reset_index()
df_tims2_non_si_2    = df_tims2_non.pivot_table(index = 'Year', columns = 'County', values = 'Non motorized serious in'                       ).reset_index()
df_tims2_non_si_2_5  = df_tims2_non.pivot_table(index = 'Year', columns = 'County', values = 'Non motorized serious in_5 Year Rolling Average').reset_index()

# Reorganize columns
df_tims2_fat_2       = df_tims2_fat_2      [cols]
df_tims2_fat_2_5     = df_tims2_fat_2_5    [cols]
df_tims2_fat100_2    = df_tims2_fat100_2   [cols]
df_tims2_fat100_2_5  = df_tims2_fat100_2_5 [cols]
df_tims2_ser_2       = df_tims2_ser_2      [cols]
df_tims2_ser_2_5     = df_tims2_ser_2_5    [cols]
df_tims2_ser100_2    = df_tims2_ser100_2   [cols]
df_tims2_ser100_2_5  = df_tims2_ser100_2_5 [cols]
df_tims2_non_fat_2   = df_tims2_non_fat_2  [cols]
df_tims2_non_fat_2_5 = df_tims2_non_fat_2_5[cols]
df_tims2_non_si_2    = df_tims2_non_si_2   [cols]
df_tims2_non_si_2_5  = df_tims2_non_si_2_5 [cols]

# View
display(df_tims2.head())

In [ ]:

## Counties
# with pd.ExcelWriter(os.path.join(path_tims, 'TIMS SWITRS Data by County.xlsx'), engine='xlsxwriter') as writer:
#     df_tims2            .to_excel(writer, index = False, sheet_name = 'All'                            )
#     df_tims2_fat_2      .to_excel(writer, index = False, sheet_name = 'Fatalities'                     )
#     df_tims2_fat_2_5    .to_excel(writer, index = False, sheet_name = 'Fatalities 5 year'              )
#     df_tims2_fat100_2   .to_excel(writer, index = False, sheet_name = 'Fatalities rate'                )
#     df_tims2_fat100_2_5 .to_excel(writer, index = False, sheet_name = 'Fatalities rate 5 year'         )
#     df_tims2_ser_2      .to_excel(writer, index = False, sheet_name = 'Serious injuries'               )
#     df_tims2_ser_2_5    .to_excel(writer, index = False, sheet_name = 'Serious injuries 5 year'        )
#     df_tims2_ser100_2   .to_excel(writer, index = False, sheet_name = 'Serious injuries rate'          )
#     df_tims2_ser100_2_5 .to_excel(writer, index = False, sheet_name = 'Serious injuries rate 5 year'   )
#     df_tims2_non_fat_2  .to_excel(writer, index = False, sheet_name = 'Non motorized fatalities'       )
#     df_tims2_non_fat_2_5.to_excel(writer, index = False, sheet_name = 'Non motorized fatalities 5 year')
#     df_tims2_non_si_2   .to_excel(writer, index = False, sheet_name = 'Non motorized serious in'       )
#     df_tims2_non_si_2_5 .to_excel(writer, index = False, sheet_name = 'Non motorized serious in 5 year')


In [ ]:
path_plots = os.path.join(path_out, 'Safety_1 Collision Rates', 'plots')
df_plot = df_tims2.copy()

df_plot = df_plot[['County', 'Year', 'Fatalities 100 mvmt', 'Serious injuries 100 mvm']]
df_plot = df_plot.rename(columns = {'Fatalities 100 mvmt':'Fatalities', 'Serious injuries 100 mvm':'Serious Injuries'})
df_plot = pd.melt(df_plot, id_vars = ['County', 'Year'], var_name = 'Category', value_name = 'value')
df_plot = df_plot.drop_duplicates()

display(df_plot.head())

fig = px.line(df_plot, x = 'Year', y = 'value', color = 'County', line_dash = 'Category', markers = False)
fig.update_layout(legend_title=None, title='TIMS Fatalities vs Serious Injuries by County (Per 1000 MVMT)')
# fig.write_html(os.path.join(path_plots, 'Safety_1_Fatalities vs Serious Injuries_line.html'))
fig.show()

df_plot = df_plot[df_plot['Year'].isin([2013, 2017, 2021])]
fig = px.bar(df_plot, x = 'Category', y = 'value', color = 'County', barmode = 'group', facet_col = 'Year')
fig.update_layout(legend_title=None, title='TIMS Fatalities vs Serious Injuries by County (Per 1000 MVMT)')
# fig.write_html(os.path.join(path_plots, 'Safety_1_Fatalities vs Serious Injuries_bar.html'))
fig.show()

In [ ]:
path_plots = os.path.join(path_out, 'Safety_2 Non-Motorized', 'plots')
df_plot = df_tims2.copy()

df_plot = df_plot[['County', 'Year', 'Non motorized fatalities', 'Non motorized serious in']]
df_plot = df_plot.rename(columns = {'Non motorized fatalities':'Fatalities', 'Non motorized serious in':'Serious Injuries'})
df_plot = pd.melt(df_plot, id_vars = ['County', 'Year'], var_name = 'Category', value_name = 'value')
df_plot = df_plot.drop_duplicates()

display(df_plot.head())

fig = px.line(df_plot, x = 'Year', y = 'value', color = 'County', line_dash = 'Category', markers = False)
fig.update_layout(legend_title=None, title='TIMS Non-Motorized Fatalities vs Serious Injuries by County')
# fig.write_html(os.path.join(path_plots, 'Safety_2_Fatalities vs Serious Injuries_line.html'))
fig.show()

df_plot = df_plot[df_plot['Year'].isin([2013, 2017, 2021])]
fig = px.bar(df_plot, x = 'Category', y = 'value', color = 'County', barmode = 'group', facet_col = 'Year')
fig.update_layout(legend_title=None, title='TIMS Non-Motorized Fatalities vs Serious Injuries by County')
# fig.write_html(os.path.join(path_plots, 'Safety_2_Fatalities vs Serious Injuries_bar.html'))
fig.show()

MPO

In [ ]:
# Import data at the MPO level
exec(open(os.path.join(path_code, 'Supplemental Scripts', 'MPO.py')).read())
df_tims3.head()

In [ ]:
# # MPO
# with pd.ExcelWriter(os.path.join(path_tims, 'TIMS SWITRS Data by MPO.xlsx'), engine='xlsxwriter') as writer:
#     df_tims3.to_excel(writer, index = False, sheet_name = 'All')

Statewide

In [ ]:
# Import data at the MPO level
exec(open(os.path.join(path_code, 'Supplemental Scripts', 'Statewide.py')).read())
df_tims4.head()

In [ ]:

# # Statewide
# with pd.ExcelWriter(os.path.join(path_tims, 'TIMS SWITRS Data Statewide.xlsx'), engine='xlsxwriter') as writer:
#     df_tims4.to_excel(writer, index = False, sheet_name = 'Statewide')

***

Safety Indicators

***

In [ ]:

# Safety_1 Collision Rates
df_safety1_1 = df_tims1[['County', 'Jurisdiction', 'Year', 'Fatalities 100 mvmt'     , 'Fatalities 100 mvmt_5 Year Rolling Average'
                                                         , 'Serious injuries 100 mvm', 'Serious injuries 100 mvm_5 Year Rolling Average']]
df_safety1_2 = df_tims2[['County', 'Year', 'Fatalities 100 mvmt'     , 'Fatalities 100 mvmt_5 Year Rolling Average'
                                         , 'Serious injuries 100 mvm', 'Serious injuries 100 mvm_5 Year Rolling Average']]
df_safety1_3 = df_tims3[['MPO', 'Year', 'Fatalities 100 mvmt'     , 'Fatalities 100 mvmt_5 Year Rolling Average'
                                      , 'Serious injuries 100 mvm', 'Serious injuries 100 mvm_5 Year Rolling Average']]
df_safety1_4 = df_tims4[['State', 'Year', 'Fatalities 100 mvmt'     , 'Fatalities 100 mvmt_5 Year Rolling Average'
                                        , 'Serious injuries 100 mvm', 'Serious injuries 100 mvm_5 Year Rolling Average']]

# Safety_2 Non-Motorized
df_safety2_1 = df_tims1[['County', 'Jurisdiction', 'Year', 'Non motorized fatalities', 'Non motorized fatalities_5 Year Rolling Average'
                                         , 'Non motorized serious in', 'Non motorized serious in_5 Year Rolling Average']]
df_safety2_2 = df_tims2[['County', 'Year', 'Non motorized fatalities', 'Non motorized fatalities_5 Year Rolling Average'
                                         , 'Non motorized serious in', 'Non motorized serious in_5 Year Rolling Average']]
df_safety2_3 = df_tims3[['MPO', 'Year', 'Non motorized fatalities', 'Non motorized fatalities_5 Year Rolling Average'
                                      , 'Non motorized serious in', 'Non motorized serious in_5 Year Rolling Average']]
df_safety2_4 = df_tims4[['State', 'Year', 'Non motorized fatalities', 'Non motorized fatalities_5 Year Rolling Average'
                                        , 'Non motorized serious in', 'Non motorized serious in_5 Year Rolling Average']]


***

Exporting

***

In [ ]:
sample_type = 'TIMS'


# # Safety_1 Collision Rates
# indicator_name = 'Safety_1'
# geography = 'Jurisdictions'
# df_about = write_about(sample_type, indicator_name, geography, year_start, year_end, path_config0)
# with pd.ExcelWriter(os.path.join(path_out, 'Safety_1 Collision Rates', 'Safety_1 Jurisdictions TIMS.xlsx'), engine='xlsxwriter') as writer:
#     df_about    .to_excel(writer, index = False, sheet_name = 'About'        , header = False)
#     df_safety1_1.to_excel(writer, index = False, sheet_name = 'Jurisdictions'                )
# geography = 'Counties'
# df_about = write_about(sample_type, indicator_name, geography, year_start, year_end, path_config0)
# with pd.ExcelWriter(os.path.join(path_out, 'Safety_1 Collision Rates', 'Safety_1 Counties TIMS.xlsx'), engine='xlsxwriter') as writer:
#     df_about    .to_excel(writer, index = False, sheet_name = 'About'   , header = False)
#     df_safety1_2.to_excel(writer, index = False, sheet_name = 'Counties'                )
# geography = 'MPO'
# df_about = write_about(sample_type, indicator_name, geography, year_start, year_end, path_config0)
# with pd.ExcelWriter(os.path.join(path_out, 'Safety_1 Collision Rates', 'Safety_1 MPO TIMS.xlsx'), engine='xlsxwriter') as writer:
#     df_about    .to_excel(writer, index = False, sheet_name = 'About', header = False)
#     df_safety1_3.to_excel(writer, index = False, sheet_name = 'MPO'                  )
# geography = 'MPO'
# df_about = write_about(sample_type, indicator_name, geography, year_start, year_end, path_config0)
# with pd.ExcelWriter(os.path.join(path_out, 'Safety_1 Collision Rates', 'Safety_1 Statewide TIMS.xlsx'), engine='xlsxwriter') as writer:
#     df_about    .to_excel(writer, index = False, sheet_name = 'About'    , header = False)
#     df_safety1_4.to_excel(writer, index = False, sheet_name = 'Statewide'                )

# # Safety_2 Non-Motorized
# indicator_name = 'Safety_2'
# geography = 'Jurisdictions'
# df_about = write_about(sample_type, indicator_name, geography, year_start, year_end, path_config0)
# with pd.ExcelWriter(os.path.join(path_out, 'Safety_2 Non-Motorized', 'Safety_2 Jurisdictions TIMS.xlsx'), engine='xlsxwriter') as writer:
#     df_about    .to_excel(writer, index = False, sheet_name = 'About'        , header = False)
#     df_safety2_1.to_excel(writer, index = False, sheet_name = 'Jurisdictions'                )
# geography = 'Counties'
# df_about = write_about(sample_type, indicator_name, geography, year_start, year_end, path_config0)
# with pd.ExcelWriter(os.path.join(path_out, 'Safety_2 Non-Motorized', 'Safety_2 Counties TIMS.xlsx'), engine='xlsxwriter') as writer:
#     df_about    .to_excel(writer, index = False, sheet_name = 'About'   , header = False)
#     df_safety2_2.to_excel(writer, index = False, sheet_name = 'Counties'                )
# geography = 'MPO'
# df_about = write_about(sample_type, indicator_name, geography, year_start, year_end, path_config0)
# with pd.ExcelWriter(os.path.join(path_out, 'Safety_2 Non-Motorized', 'Safety_2 MPO TIMS.xlsx'), engine='xlsxwriter') as writer:
#     df_about    .to_excel(writer, index = False, sheet_name = 'About', header = False)
#     df_safety2_3.to_excel(writer, index = False, sheet_name = 'MPO'                  )
# geography = 'MPO'
# df_about = write_about(sample_type, indicator_name, geography, year_start, year_end, path_config0)
# with pd.ExcelWriter(os.path.join(path_out, 'Safety_2 Non-Motorized', 'Safety_2 Statewide TIMS.xlsx'), engine='xlsxwriter') as writer:
#     df_about    .to_excel(writer, index = False, sheet_name = 'About'    , header = False)
#     df_safety2_4.to_excel(writer, index = False, sheet_name = 'Statewide'                )
    

In [ ]:
df_safety1_3['State'] = 'CA'
df_safety1_4['MPO'  ] = 'Statewide'

df_safety_1 = pd.concat([df_safety1_3, df_safety1_4])
df_safety_1 = df_safety_1.drop('State', axis = 1)
df_safety_1 = df_safety_1.rename(columns = {'MPO':'Group'})

df_safety2_3['State'] = 'CA'
df_safety2_4['MPO'  ] = 'Statewide'

df_safety_2 = pd.concat([df_safety2_3, df_safety2_4])
df_safety_2 = df_safety_2.drop('State', axis = 1)
df_safety_2 = df_safety_2.rename(columns = {'MPO':'Group'})


# df_safety_1.to_csv(os.path.join(path_agol, 'Safety_1', 'Safety_1 MPO TIMS.csv'), index = False)
# df_safety_2.to_csv(os.path.join(path_agol, 'Safety_2', 'Safety_2 MPO TIMS.csv'), index = False)


df_safety_2